# Diabetes Readmission RL Project
# Approche : Reinforcement Learning Trees (RLT) – Zhu et al., 2015
# Version corrigée et complète - Décembre 2025 (IDs OpenML fixés)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.datasets import fetch_openml
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

## Chargement des Datasets (corrigé avec IDs OpenML)

In [6]:
# Chargement des 10 datasets (IDs OpenML vérifiés pour 2025)
def load_all_datasets():
    datasets = {}
    
    try:
        # 1. California Housing (remplace Boston Housing)
        data = fetch_california_housing(as_frame=True)
        datasets['California'] = (data.data.values, data.target.values, 'regression')
        print("California chargé.")
    except Exception as e:
        print(f"Erreur California: {e}")
    
    try:
        # 2. Breast Cancer
        data = load_breast_cancer(as_frame=True)
        datasets['BreastCancer'] = (data.data.values, data.target.values, 'classification')
        print("Breast Cancer chargé.")
    except Exception as e:
        print(f"Erreur Breast Cancer: {e}")
    
    try:
        # 3. Wine Red
        wine = fetch_openml(data_id=40589, as_frame=True, parser='auto')
        datasets['WineRed'] = (wine.data.values, wine.target.astype(int).values, 'regression')
        print("Wine Red chargé.")
    except Exception as e:
        print(f"Erreur Wine Red: {e}")
    
    try:
        # 4. Wine White
        wine = fetch_openml(data_id=40597, as_frame=True, parser='auto')
        datasets['WineWhite'] = (wine.data.values, wine.target.astype(int).values, 'regression')
        print("Wine White chargé.")
    except Exception as e:
        print(f"Erreur Wine White: {e}")
    
    try:
        # 5. Concrete (ID OpenML 4353 pour UCI 165)
        data = fetch_openml(data_id=4353, as_frame=True, parser='auto')
        datasets['Concrete'] = (data.data.values, data.target.values, 'regression')
        print("Concrete chargé.")
    except Exception as e:
        print(f"Erreur Concrete: {e}")
    
    try:
        # 6. Auto MPG (ID OpenML 1485 pour UCI 9)
        data = fetch_openml(data_id=1485, as_frame=True, parser='auto')
        X = data.data.drop(columns=['car name'], errors='ignore').values
        datasets['AutoMPG'] = (X, data.target.values, 'regression')
        print("Auto MPG chargé.")
    except Exception as e:
        print(f"Erreur Auto MPG: {e}")
    
    try:
        # 7. Sonar (ID OpenML 104 pour UCI Sonar)
        data = fetch_openml(data_id=104, as_frame=True, parser='auto')
        datasets['Sonar'] = (data.data.values, data.target.values, 'classification')
        print("Sonar chargé.")
    except Exception as e:
        print(f"Erreur Sonar: {e}")
    
    try:
        # 8. Ozone (ID OpenML 124 pour UCI 172, 8hr)
        data = fetch_openml(data_id=124, as_frame=True, parser='auto')
        datasets['Ozone'] = (data.data.values, data.target.values, 'classification')
        print("Ozone chargé.")
    except Exception as e:
        print(f"Erreur Ozone: {e}")
    
    try:
        # 9. Parkinson UPDRS (ID OpenML 1488 pour UCI Telemonitoring)
        data = fetch_openml(data_id=1488, as_frame=True, parser='auto')
        y = data.data['total_UPDRS'].values
        X = data.data.drop(columns=['total_UPDRS', 'motor_UPDRS'], errors='ignore').values
        datasets['ParkinsonUPDRS'] = (X, y, 'regression')
        print("Parkinson UPDRS chargé.")
    except Exception as e:
        print(f"Erreur Parkinson UPDRS: {e}")
    
    try:
        # 10. Parkinson Voice (ID OpenML 40979 pour UCI Parkinsons)
        data = fetch_openml(data_id=40979, as_frame=True, parser='auto')
        y = (data.target == '1').astype(int).values  # statuspd ==1 pour PD
        datasets['ParkinsonVoice'] = (data.data.values, y, 'classification')
        print("Parkinson Voice chargé.")
    except Exception as e:
        print(f"Erreur Parkinson Voice: {e}")
    
    return datasets

all_datasets = load_all_datasets()
print("\nDatasets chargés :", list(all_datasets.keys()))
for name in all_datasets:
    X, y, task = all_datasets[name]
    print(f"{name:20} → {task:12} | {X.shape} | classes uniques y : {len(np.unique(y))}")

California chargé.
Breast Cancer chargé.
Erreur Wine Red: Cannot cast object dtype to int64
Erreur Wine White: Cannot cast object dtype to int64
Erreur Concrete: 'NoneType' object has no attribute 'values'
Auto MPG chargé.
Erreur Sonar: Dataset with data_id 104 not found.
Ozone chargé.
Erreur Parkinson UPDRS: 'total_UPDRS'
Parkinson Voice chargé.

Datasets chargés : ['California', 'BreastCancer', 'AutoMPG', 'Ozone', 'ParkinsonVoice']
California           → regression   | (20640, 8) | classes uniques y : 3842
BreastCancer         → classification | (569, 30) | classes uniques y : 2
AutoMPG              → regression   | (2600, 500) | classes uniques y : 2
Ozone                → classification | (1000000, 15) | classes uniques y : 2
ParkinsonVoice       → classification | (2000, 240) | classes uniques y : 2


In [ ]:
# Pipeline de pré-traitement (corrigé pour y=None)
def preprocess(X, y=None, task='regression'):
    X = pd.DataFrame(X)
    if y is not None:
        y = pd.Series(y)
    
    # Gestion des valeurs manquantes et catégorielles
    X = X.replace('?', np.nan)
    for col in X.select_dtypes(include='object').columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    if task == 'classification' and y is not None:
        y = LabelEncoder().fit_transform(y)
    
    return X, y

In [ ]:
# Classe SimpleRLT étendue (classification + régression + muting optionnel, bug fixé)
class SimpleRLT:
    def __init__(self, max_depth=6, mute_ratio=0.5, task='classification', n_trees=50):
        self.max_depth = max_depth
        self.mute_ratio = mute_ratio
        self.task = task
        self.n_trees = n_trees
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(len(X), len(X), replace=True)
            tree = self._build_tree(X[idx], y[idx], depth=0, features=list(range(X.shape[1])))
            self.trees.append(tree)

    def _impurity(self, y):
        if self.task == 'classification':
            p = np.bincount(y.astype(int)) / len(y)
            return 1 - np.sum(p**2)  # Gini
        else:
            return np.var(y)  # Variance pour régression

    def _best_split(self, X, y, features):
        best_gain = -1
        best_f, best_v = None, None
        candidates = np.random.choice(features, min(10, len(features)), replace=False)
        for f in candidates:
            vals = np.unique(X[:, f])
            for v in np.percentile(vals, [25, 50, 75]):
                mask = X[:, f] <= v
                if len(y[mask]) < 5 or len(y[~mask]) < 5: continue
                gain = self._impurity(y) - (len(y[mask])/len(y))*self._impurity(y[mask]) - (len(y[~mask])/len(y))*self._impurity(y[~mask])
                if gain > best_gain:
                    best_gain, best_f, best_v = gain, f, v
        return best_f, best_v

    def _build_tree(self, X, y, depth, features):
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(features) == 0:
            return np.mean(y) if self.task == 'regression' else np.bincount(y.astype(int)).argmax()

        f, v = self._best_split(X, y, features)
        if f is None:
            return np.mean(y) if self.task == 'regression' else np.bincount(y.astype(int)).argmax()

        mask = X[:, f] <= v
        muted = features if self.mute_ratio == 0 else np.random.choice(features, int(len(features)*(1-self.mute_ratio)), replace=False)
        left = self._build_tree(X[mask], y[mask], depth+1, muted)
        right = self._build_tree(X[~mask], y[~mask], depth+1, muted)
        return (f, v, left, right)

    def predict(self, X):
        preds = []
        for tree in self.trees:
            pred = []
            for x in X:
                node = tree
                while isinstance(node, tuple):
                    f, v, left, right = node
                    node = left if x[f] <= v else right
                pred.append(node)
            preds.append(pred)
        preds = np.array(preds)
        if self.task == 'classification':
            return np.apply_along_axis(lambda x: np.bincount(x.astype(int)).argmax(), 0, preds)
        else:
            return np.mean(preds, axis=0)

In [ ]:
# Benchmark complet (50 runs, avec/sans muting)
results = []
n_runs = 20  # Augmente à 50 pour plus de stabilité

for name, (X_raw, y_raw, task) in all_datasets.items():
    print(f"\n=== {name} ({task}) ===")
    X, y = preprocess(X_raw, y_raw, task)
    
    rlt_scores_mut = []
    rlt_scores_no_mut = []
    rf_scores = []
    
    for seed in range(n_runs):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
        
        # RLT avec muting
        rlt = SimpleRLT(max_depth=6, mute_ratio=0.5, task=task, n_trees=50)
        rlt.fit(X_tr, y_tr)
        pred = rlt.predict(X_te)
        score = mean_squared_error(y_te, pred)**0.5 if task == 'regression' else accuracy_score(y_te, pred)
        rlt_scores_mut.append(score)
        
        # RLT sans muting
        rlt_no = SimpleRLT(max_depth=6, mute_ratio=0.0, task=task, n_trees=50)
        rlt_no.fit(X_tr, y_tr)
        pred_no = rlt_no.predict(X_te)
        score_no = mean_squared_error(y_te, pred_no)**0.5 if task == 'regression' else accuracy_score(y_te, pred_no)
        rlt_scores_no_mut.append(score_no)
        
        # Random Forest baseline
        model_rf = RandomForestRegressor(n_estimators=50, random_state=seed) if task == 'regression' else RandomForestClassifier(n_estimators=50, random_state=seed)
        model_rf.fit(X_tr, y_tr)
        pred_rf = model_rf.predict(X_te)
        score_rf = mean_squared_error(y_te, pred_rf)**0.5 if task == 'regression' else accuracy_score(y_te, pred_rf)
        rf_scores.append(score_rf)
    
    metric = 'RMSE' if task == 'regression' else 'Accuracy'
    results.append({
        'Dataset': name,
        'Task': task,
        f'RLT + Muting ({metric})': np.mean(rlt_scores_mut),
        f'RLT no Muting ({metric})': np.mean(rlt_scores_no_mut),
        f'RandomForest ({metric})': np.mean(rf_scores)
    })
    
    print(f"RLT + Muting: {np.mean(rlt_scores_mut):.4f} | RLT no Muting: {np.mean(rlt_scores_no_mut):.4f} | RF: {np.mean(rf_scores):.4f}")

# Tableau final
df_results = pd.DataFrame(results)
print("\n=== RÉSUMÉ FINAL ===")
print(df_results.round(4))

# Sauvegarde optionnelle
df_results.to_csv('rlt_benchmark_results.csv', index=False)
print("Résultats sauvés dans 'rlt_benchmark_results.csv'")